# Member 2 — How the tracking model works

This notebook **runs** Component 2 on a retail clip and shows the live result: YOLOv8 person boxes, ByteTrack IDs, and Kalman trajectories.

```text
Video frame → YOLOv8 (person) → ByteTrack + Kalman → ID + bbox + path
```

- **YOLOv8** finds people in each frame (anchor-free boxes).
- **ByteTrack** keeps the same `person_id` by matching high-score boxes first, then low-score boxes during occlusions.
- **Kalman filter** predicts where a person should be if a box is missing for a short time.
- This component does **not** label shoplifting. That is Member 3.

Open this file in Cursor/VS Code and run all cells, or use the generated previews under `outputs/demo_preview/`.

In [ ]:
from collections import defaultdict, deque
from pathlib import Path
import json
import sys

import cv2
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "src" / "detector.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.detector import PersonDetector, resolve_weights
from src.pipeline import load_tracker_settings, make_tracker
from src.visualize import draw_tracks

VIDEO = ROOT / "Dataset" / "videos" / "not_shoplifting1.mp4"
WEIGHTS = resolve_weights(ROOT / "models" / "yolov8n-person-track.pt")
PREVIEW = ROOT / "outputs" / "demo_preview"
OUT_VIDEO = ROOT / "outputs" / "demo_how_model_works.mp4"
OUT_JSON = ROOT / "outputs" / "demo_how_model_works.json"
MAX_FRAMES = 90

print("project:", ROOT)
print("weights:", WEIGHTS)
print("video:", VIDEO, "exists=", VIDEO.exists())

In [ ]:
detector = PersonDetector(WEIGHTS, confidence=0.25)
tracker = make_tracker(load_tracker_settings())
capture = cv2.VideoCapture(str(VIDEO))
if not capture.isOpened():
    raise FileNotFoundError(VIDEO)

fps = capture.get(cv2.CAP_PROP_FPS) or 25
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
PREVIEW.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(str(OUT_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

trails = defaultdict(lambda: deque(maxlen=30))
sequence = []
snapshots = {}
wanted = {0, 15, 30, 60}

for frame_id in range(MAX_FRAMES):
    ok, frame = capture.read()
    if not ok:
        break
    detections = detector.detect(frame)
    tracks = [track.to_contract() for track in tracker.update(detections)]
    sequence.append({"frame_id": frame_id, "tracks": tracks})
    vis = draw_tracks(frame.copy(), tracks, trails)
    writer.write(vis)
    if frame_id in wanted:
        snapshots[frame_id] = vis.copy()
        cv2.imwrite(str(PREVIEW / f"frame_{frame_id:03d}.jpg"), vis)

capture.release()
writer.release()
OUT_JSON.write_text(json.dumps(sequence, indent=2), encoding="utf-8")

ids = sorted({t["person_id"] for frame in sequence for t in frame["tracks"]})
print(f"processed {len(sequence)} frames at {width}x{height}")
print(f"unique person_id values: {ids}")
print(f"annotated video: {OUT_VIDEO}")
print(f"tracking JSON: {OUT_JSON}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, frame_id in zip(axes.ravel(), sorted(snapshots)):
    rgb = cv2.cvtColor(snapshots[frame_id], cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    n = len(sequence[frame_id]["tracks"]) if frame_id < len(sequence) else 0
    ax.set_title(f"frame {frame_id} — {n} tracked people")
    ax.axis("off")
plt.suptitle("YOLOv8 + ByteTrack: same colour/ID follows the same person")
plt.tight_layout()
plt.show()

In [ ]:
print("Sample tracking JSON (Member 3 contract):")
print(json.dumps(sequence[min(15, len(sequence)-1)], indent=2)[:1200])

## Saved demo output (already run on this PC)

The model was executed on `Dataset/videos/not_shoplifting1.mp4` using `models/yolov8n-person-track.pt`.

| Item | Path |
| --- | --- |
| Annotated video | `outputs/demo_how_model_works.mp4` |
| Tracking JSON | `outputs/demo_how_model_works.json` |
| Preview frames | `outputs/demo_preview/frame_000.jpg` … `frame_060.jpg` |

**What you should see:** coloured boxes with `ID 1`, `ID 2`, `ID 3` (and later `ID 4`). The same ID stays on the same person as they move. Low-score / partial people can still keep an ID because ByteTrack uses the second association pass + Kalman prediction.

To re-run: open this notebook, set the kernel to Python 3, **Run All**.